# Final Test Evaluation — Frozen Configuration

Notebook ini menghitung metrik **final test** dari:

- `manual_qrels_test_v2_REVIEWED.csv`
- `test_ranking_final_config_PRIVATE.csv`

Konfigurasi sistem **tidak diubah lagi**. Notebook ini hanya mengevaluasi ranking yang sudah dibekukan pada tahap sebelumnya.

Metrik:
- Hit@5 dan Hit@10 untuk relevance grade ≥ 1
- Strict Hit@5 dan Hit@10 untuk relevance grade = 2
- MRR untuk grade ≥ 1
- Strict MRR untuk grade = 2
- nDCG@5 dan nDCG@10 dengan graded relevance 0/1/2

Tidak ada tuning pada test set.


In [1]:
# ============================================================
# FINAL TEST EVALUATION
# Frozen ranking + reviewed manual qrels
# ============================================================

import pandas as pd
import numpy as np
import math

QRELS_PATH = "manual_qrels_test_v2_REVIEWED.csv"
RANKING_PATH = "test_ranking_final_config_PRIVATE.csv"

SUMMARY_OUT = "hasil_final_test_ringkas.csv"
PER_QUERY_OUT = "hasil_final_test_per_query.csv"
DETAIL_OUT = "hasil_final_test_detail.csv"

TOP_KS = [5, 10]


def dcg_at_k(grades, k):
    value = 0.0
    for i, rel in enumerate(grades[:k], start=1):
        gain = (2 ** int(rel)) - 1
        value += gain / math.log2(i + 1)
    return value


def ndcg_at_k(grades, all_judged_grades, k):
    dcg = dcg_at_k(grades, k)
    ideal = sorted([int(x) for x in all_judged_grades], reverse=True)
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg <= 0 else dcg / idcg


def reciprocal_rank(grades, threshold):
    for i, grade in enumerate(grades, start=1):
        if int(grade) >= threshold:
            return 1.0 / i
    return 0.0


def hit_at_k(grades, k, threshold):
    return int(any(int(g) >= threshold for g in grades[:k]))


# ------------------------------------------------------------
# 1. Load
# ------------------------------------------------------------
qrels = pd.read_csv(QRELS_PATH)
ranking = pd.read_csv(RANKING_PATH)

required_qrels = {
    "query_id",
    "query_text",
    "candidate_id",
    "reviewed_relevance_grade",
}
required_ranking = {
    "query_id",
    "query_text",
    "rank",
    "candidate_id",
    "candidate_title",
}

missing_qrels = required_qrels - set(qrels.columns)
missing_ranking = required_ranking - set(ranking.columns)

if missing_qrels:
    raise ValueError(f"Kolom qrels hilang: {sorted(missing_qrels)}")
if missing_ranking:
    raise ValueError(f"Kolom ranking hilang: {sorted(missing_ranking)}")

# Numeric grade validation
qrels["reviewed_relevance_grade"] = pd.to_numeric(
    qrels["reviewed_relevance_grade"],
    errors="coerce",
)

if qrels["reviewed_relevance_grade"].isna().any():
    n_missing = int(qrels["reviewed_relevance_grade"].isna().sum())
    raise ValueError(f"Ada {n_missing} qrels tanpa reviewed grade.")

qrels["reviewed_relevance_grade"] = qrels["reviewed_relevance_grade"].astype(int)

grades_unique = set(qrels["reviewed_relevance_grade"].unique())
if not grades_unique.issubset({0, 1, 2}):
    raise ValueError(f"Grade di luar 0/1/2 ditemukan: {sorted(grades_unique)}")

# Duplicate validation
qrel_dup = qrels.duplicated(["query_id", "candidate_id"]).sum()
rank_dup = ranking.duplicated(["query_id", "candidate_id"]).sum()

if qrel_dup:
    raise ValueError(f"Qrels memiliki {qrel_dup} duplicate query-candidate.")
if rank_dup:
    raise ValueError(f"Ranking memiliki {rank_dup} duplicate query-candidate.")

# Per query rank count and order checks
rank_counts = ranking.groupby("query_id").size()
if not (rank_counts == 10).all():
    bad = rank_counts[rank_counts != 10]
    raise ValueError(
        "Tidak semua query memiliki tepat 10 ranking. "
        f"Contoh masalah: {bad.head().to_dict()}"
    )

for qid, grp in ranking.groupby("query_id"):
    ranks = sorted(grp["rank"].astype(int).tolist())
    if ranks != list(range(1, 11)):
        raise ValueError(f"Rank query {qid} bukan 1..10 lengkap: {ranks}")

print("VALIDASI DATA FINAL TEST")
print(f"Qrels judgments       : {len(qrels)}")
print(f"Ranking rows          : {len(ranking)}")
print(f"Test queries          : {ranking['query_id'].nunique()}")
print(f"Qrels duplicate       : {qrel_dup}")
print(f"Ranking duplicate     : {rank_dup}")
print(
    "Distribusi reviewed grade:",
    qrels["reviewed_relevance_grade"].value_counts().sort_index().to_dict(),
)


# ------------------------------------------------------------
# 2. Join ranking to reviewed qrels
# ------------------------------------------------------------
joined = ranking.merge(
    qrels[
        [
            "query_id",
            "query_text",
            "candidate_id",
            "reviewed_relevance_grade",
        ]
    ],
    on=["query_id", "query_text", "candidate_id"],
    how="left",
    validate="one_to_one",
)

unjudged = joined["reviewed_relevance_grade"].isna().sum()

if unjudged:
    examples = joined[joined["reviewed_relevance_grade"].isna()][
        ["query_id", "rank", "candidate_id", "candidate_title"]
    ].head(10)

    raise ValueError(
        f"Ada {unjudged} ranking rows tanpa manual judgment.\n"
        f"Contoh:\n{examples.to_string(index=False)}"
    )

joined["reviewed_relevance_grade"] = joined["reviewed_relevance_grade"].astype(int)

joined = joined.sort_values(["query_id", "rank"]).reset_index(drop=True)

print(f"Unjudged ranked candidates: {unjudged}")


# ------------------------------------------------------------
# 3. All judged grades per query for IDCG
# ------------------------------------------------------------
qrels_by_query = (
    qrels.groupby("query_id")["reviewed_relevance_grade"]
    .apply(lambda x: [int(v) for v in x.tolist()])
    .to_dict()
)


# ------------------------------------------------------------
# 4. Per-query metrics
# ------------------------------------------------------------
per_query_rows = []

for qid, grp in joined.groupby("query_id", sort=True):
    grp = grp.sort_values("rank")
    grades = grp["reviewed_relevance_grade"].astype(int).tolist()
    all_judged = qrels_by_query[qid]

    row = {
        "query_id": qid,
        "query_text": grp["query_text"].iloc[0],
        "hit_at_5_grade_ge_1": hit_at_k(grades, 5, threshold=1),
        "hit_at_10_grade_ge_1": hit_at_k(grades, 10, threshold=1),
        "strict_hit_at_5_grade_2": hit_at_k(grades, 5, threshold=2),
        "strict_hit_at_10_grade_2": hit_at_k(grades, 10, threshold=2),
        "mrr_grade_ge_1": reciprocal_rank(grades, threshold=1),
        "strict_mrr_grade_2": reciprocal_rank(grades, threshold=2),
        "ndcg_at_5": ndcg_at_k(grades, all_judged, 5),
        "ndcg_at_10": ndcg_at_k(grades, all_judged, 10),
        "top10_grades": " | ".join(str(g) for g in grades),
        "top10_titles": " | ".join(grp["candidate_title"].astype(str).tolist()),
    }

    per_query_rows.append(row)


per_query_df = pd.DataFrame(per_query_rows)


# ------------------------------------------------------------
# 5. Aggregate final metrics
# ------------------------------------------------------------
summary = {
    "n_query": len(per_query_df),
    "hit_at_5_grade_ge_1": float(per_query_df["hit_at_5_grade_ge_1"].mean()),
    "hit_at_10_grade_ge_1": float(per_query_df["hit_at_10_grade_ge_1"].mean()),
    "strict_hit_at_5_grade_2": float(per_query_df["strict_hit_at_5_grade_2"].mean()),
    "strict_hit_at_10_grade_2": float(per_query_df["strict_hit_at_10_grade_2"].mean()),
    "mrr_grade_ge_1": float(per_query_df["mrr_grade_ge_1"].mean()),
    "strict_mrr_grade_2": float(per_query_df["strict_mrr_grade_2"].mean()),
    "ndcg_at_5": float(per_query_df["ndcg_at_5"].mean()),
    "ndcg_at_10": float(per_query_df["ndcg_at_10"].mean()),
}

summary_df = pd.DataFrame([summary])


# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------
summary_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)

per_query_df.to_csv(
    PER_QUERY_OUT,
    index=False,
    encoding="utf-8-sig",
)

joined.to_csv(
    DETAIL_OUT,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 7. Print final result
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("HASIL FINAL TEST - FROZEN CONFIGURATION")
print("=" * 100)

print(f"Total test query              : {summary['n_query']}")
print(f"Hit@5 (grade>=1)             : {summary['hit_at_5_grade_ge_1']:.4f}")
print(f"Hit@10 (grade>=1)            : {summary['hit_at_10_grade_ge_1']:.4f}")
print(f"Strict Hit@5 (grade=2)       : {summary['strict_hit_at_5_grade_2']:.4f}")
print(f"Strict Hit@10 (grade=2)      : {summary['strict_hit_at_10_grade_2']:.4f}")
print(f"MRR (grade>=1)               : {summary['mrr_grade_ge_1']:.4f}")
print(f"Strict MRR (grade=2)         : {summary['strict_mrr_grade_2']:.4f}")
print(f"nDCG@5                       : {summary['ndcg_at_5']:.4f}")
print(f"nDCG@10                      : {summary['ndcg_at_10']:.4f}")

print("\nFINAL CONFIGURATION (FROZEN)")
print("- Query: BASE_V2 / query_text_v2")
print("- Retrieval: SBERT top-50 + keyword extra max-20")
print("- Semantic score: min-max normalized per candidate pool")
print("- WLC: 0.7 semantic + 0.3 harmonic ingredient efficiency")
print("- Test set: 90 query")
print("- Relevance judgments: manual reviewed grades 0/1/2")

print("\nOUTPUT:")
print(f"- {SUMMARY_OUT}")
print(f"- {PER_QUERY_OUT}")
print(f"- {DETAIL_OUT}")

print("\nCATATAN:")
print("- Ini adalah evaluasi final test, bukan validation.")
print("- Tidak ada tuning parameter setelah hasil test ini dilihat.")

VALIDASI DATA FINAL TEST
Qrels judgments       : 900
Ranking rows          : 900
Test queries          : 90
Qrels duplicate       : 0
Ranking duplicate     : 0
Distribusi reviewed grade: {0: 15, 1: 378, 2: 507}
Unjudged ranked candidates: 0

HASIL FINAL TEST - FROZEN CONFIGURATION
Total test query              : 90
Hit@5 (grade>=1)             : 1.0000
Hit@10 (grade>=1)            : 1.0000
Strict Hit@5 (grade=2)       : 0.8667
Strict Hit@10 (grade=2)      : 0.9000
MRR (grade>=1)               : 1.0000
Strict MRR (grade=2)         : 0.7354
nDCG@5                       : 0.8644
nDCG@10                      : 0.9358

FINAL CONFIGURATION (FROZEN)
- Query: BASE_V2 / query_text_v2
- Retrieval: SBERT top-50 + keyword extra max-20
- Semantic score: min-max normalized per candidate pool
- WLC: 0.7 semantic + 0.3 harmonic ingredient efficiency
- Test set: 90 query
- Relevance judgments: manual reviewed grades 0/1/2

OUTPUT:
- hasil_final_test_ringkas.csv
- hasil_final_test_per_query.csv
- hasil_